# 2D FP32 Training Export v0.3

This notebook is orchestration-only for the directory-based 2D export pipeline.

Flow:
1. Select `input_dir` and `output_root`
2. Build `ExportConfig`
3. Run `export_directory(config)`
4. Preview output manifests


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display
import ipywidgets as widgets
from ipyfilechooser import FileChooser


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'preprocessing' / 'export_2d_training.py').exists():
            return candidate
    raise RuntimeError('Could not find repo root containing preprocessing/export_2d_training.py')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from preprocessing.export_2d_training import ExportConfig, export_directory

display(Markdown(f'Using repo root: `{REPO_ROOT}`'))


Using repo root: `/home/mitch/development/raccoon-ball`

In [ ]:
DEFAULT_INPUT_DIR = REPO_ROOT / 'datasets'
DEFAULT_OUTPUT_ROOT = REPO_ROOT / 'preprocessing' / '03.training-export' / 'output'

BACKEND = 'torch-cuda'  # Change to 'cpu' for debug fallback
SKIP_EXISTING = True

# Smoke-test controls
SOURCE_FILES = None  # Example: [DEFAULT_INPUT_DIR / 'normal/example.wav', DEFAULT_INPUT_DIR / 'abnormal/example.wav']
MAX_FILES = None     # Example: 2

# Feature / export configuration
TARGET_SR = 16000
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
WINDOW = 'hann'
NUM_BANDS = 96
FMIN = 50.0
FMAX = 8000.0
LOG_POWER_REFERENCE = 1.0
EPSILON = 1e-8
ENERGY_BINS = 24
LOW_ENERGY_FLOOR = 0.0
MIN_ACTIVE_BANDS_PER_FRAME = 0
WINDOW_FRAMES = 64
STRIDE_FRAMES = 32


## Select Input and Output Directories
If no explicit selection is made, defaults are used.


In [3]:
def _chooser_start_dir(path: Path) -> Path:
    if path.exists():
        return path
    for candidate in [path.parent, REPO_ROOT, Path.cwd()]:
        if candidate.exists():
            return candidate
    return Path.cwd()


input_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_INPUT_DIR)))
input_dir_chooser.title = '<b>Input directory (recursive .wav scan)</b>'
input_dir_chooser.show_only_dirs = True
input_dir_chooser.use_dir_icons = True
input_dir_chooser.select_default = True

output_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_OUTPUT_ROOT)))
output_dir_chooser.title = '<b>Output root directory</b>'
output_dir_chooser.show_only_dirs = True
output_dir_chooser.use_dir_icons = True
output_dir_chooser.select_default = True

display(
    widgets.HBox([
        widgets.VBox([input_dir_chooser]),
        widgets.VBox([output_dir_chooser]),
    ])
)


In [4]:
def resolve_selected_dir(chooser: FileChooser, fallback: Path) -> Path:
    selected = getattr(chooser, 'selected', None)
    if selected:
        return Path(selected).expanduser().resolve()
    return fallback.expanduser().resolve()


input_dir = resolve_selected_dir(input_dir_chooser, DEFAULT_INPUT_DIR)
output_root = resolve_selected_dir(output_dir_chooser, DEFAULT_OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)

display(Markdown(f'- `input_dir`: `{input_dir}`\n- `output_root`: `{output_root}`'))


- `input_dir`: `/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal`
- `output_root`: `/home/mitch/development/raccoon-ball/preprocessing/03.training-export/output/-6db-pump/id_00`

In [5]:
wav_candidates = sorted(p for p in input_dir.rglob('*') if p.is_file() and p.suffix.lower() == '.wav')
if MAX_FILES is None:
    selected_preview = wav_candidates
else:
    selected_preview = wav_candidates[:MAX_FILES]

print(f'Total .wav files discovered under input_dir: {len(wav_candidates)}')
print(f'Files selected for this run (after MAX_FILES filter): {len(selected_preview)}')
for path in selected_preview[:10]:
    print(path)

if not selected_preview and SOURCE_FILES is None:
    raise RuntimeError(
        f'No .wav files found under input_dir={input_dir}. '
        'Pick a directory that contains WAV files or set SOURCE_FILES explicitly.'
    )


Total .wav files discovered under input_dir: 1006
Files selected for this run (after MAX_FILES filter): 10
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000000.wav
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000001.wav
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000002.wav
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000003.wav
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000004.wav
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000005.wav
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000006.wav
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000007.wav
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000008.wav
/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal/00000009.wav


In [6]:
config = ExportConfig(
    input_dir=input_dir,
    output_root=output_root,
    backend=BACKEND,
    source_files=SOURCE_FILES,
    max_files=MAX_FILES,
    skip_existing=SKIP_EXISTING,
    target_sr=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    window=WINDOW,
    num_bands=NUM_BANDS,
    fmin=FMIN,
    fmax=FMAX,
    log_power_reference=LOG_POWER_REFERENCE,
    epsilon=EPSILON,
    normalize_band_energy=True,
    normalization_mode='per_clip_minmax',
    energy_bins=ENERGY_BINS,
    low_energy_floor=LOW_ENERGY_FLOOR,
    min_active_bands_per_frame=MIN_ACTIVE_BANDS_PER_FRAME,
    window_frames=WINDOW_FRAMES,
    stride_frames=STRIDE_FRAMES,
)
config


ExportConfig(input_dir=PosixPath('/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal'), output_root=PosixPath('/home/mitch/development/raccoon-ball/preprocessing/03.training-export/output/-6db-pump/id_00'), backend='torch-cuda', source_files=None, max_files=10, skip_existing=True, target_sr=16000, mono=True, audio_offset_seconds=0.0, max_audio_seconds=None, n_fft=1024, hop_length=256, win_length=1024, window='hann', num_bands=96, fmin=50.0, fmax=8000.0, log_power_reference=1.0, epsilon=1e-08, normalize_band_energy=True, normalization_mode='per_clip_minmax', energy_bins=24, low_energy_floor=0.0, min_active_bands_per_frame=0, window_frames=64, stride_frames=32, manifest_prefix=None)

In [7]:
summary = export_directory(config)
summary


{'run_id': '20260319_092602',
 'run_started_utc': '2026-03-19T09:26:02.412986+00:00',
 'run_finished_utc': '2026-03-19T09:26:03.680267+00:00',
 'input_dir': '/home/mitch/development/raccoon-ball/datasets/-6db-pump/id_00/normal',
 'output_root': '/home/mitch/development/raccoon-ball/preprocessing/03.training-export/output/-6db-pump/id_00',
 'backend': 'torch-cuda',
 'device': 'cuda',
 'num_source_files_selected': 10,
 'num_file_rows': 10,
 'num_window_rows': 180,
 'status_counts': {'exported': 10},
 'files_manifest_path': '/home/mitch/development/raccoon-ball/preprocessing/03.training-export/output/-6db-pump/id_00/manifests/20260319_092602_files.parquet',
 'windows_manifest_path': '/home/mitch/development/raccoon-ball/preprocessing/03.training-export/output/-6db-pump/id_00/manifests/20260319_092602_windows.parquet',
 'config_snapshot_path': '/home/mitch/development/raccoon-ball/preprocessing/03.training-export/output/-6db-pump/id_00/manifests/20260319_092602_config.json'}

In [8]:
files_manifest_path = Path(summary['files_manifest_path'])
windows_manifest_path = Path(summary['windows_manifest_path'])

files_df = pd.read_parquet(files_manifest_path)
windows_df = pd.read_parquet(windows_manifest_path)

display(Markdown('### Run Summary'))
display(pd.DataFrame([summary]))

display(Markdown('### Files Manifest (sample)'))
display(files_df.head(20))

display(Markdown('### Windows Manifest (sample)'))
display(windows_df.head(20))


### Run Summary

,run_id,run_started_utc,run_finished_utc,input_dir,output_root,backend,device,num_source_files_selected,num_file_rows,num_window_rows,status_counts,files_manifest_path,windows_manifest_path,config_snapshot_path
0,20260319_092602,2026-03-19T09:26:02.412986+00:00,2026-03-19T09:26:03.680267+00:00,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/preproces...,torch-cuda,cuda,10,10,180,{'exported': 10},/home/mitch/development/raccoon-ball/preproces...,/home/mitch/development/raccoon-ball/preproces...,/home/mitch/development/raccoon-ball/preproces...


### Files Manifest (sample)

,source_file,relative_source_path,tensor_npz_path,status,error,source_sample_rate,target_sample_rate,num_samples_target_sr,num_frames,num_windows,audio_loader,backend,device,started_utc,finished_utc
0,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:02.413259+00:00,2026-03-19T09:26:03.163309+00:00
1,/home/mitch/development/raccoon-ball/datasets/...,00000001.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:03.163608+00:00,2026-03-19T09:26:03.216955+00:00
2,/home/mitch/development/raccoon-ball/datasets/...,00000002.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:03.217229+00:00,2026-03-19T09:26:03.271554+00:00
3,/home/mitch/development/raccoon-ball/datasets/...,00000003.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:03.271837+00:00,2026-03-19T09:26:03.324558+00:00
4,/home/mitch/development/raccoon-ball/datasets/...,00000004.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:03.324833+00:00,2026-03-19T09:26:03.377844+00:00
5,/home/mitch/development/raccoon-ball/datasets/...,00000005.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:03.378110+00:00,2026-03-19T09:26:03.430978+00:00
6,/home/mitch/development/raccoon-ball/datasets/...,00000006.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:03.431251+00:00,2026-03-19T09:26:03.483937+00:00
7,/home/mitch/development/raccoon-ball/datasets/...,00000007.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:03.484225+00:00,2026-03-19T09:26:03.537210+00:00
8,/home/mitch/development/raccoon-ball/datasets/...,00000008.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:03.537478+00:00,2026-03-19T09:26:03.590813+00:00
9,/home/mitch/development/raccoon-ball/datasets/...,00000009.wav,/home/mitch/development/raccoon-ball/preproces...,exported,None,16000,16000,160000,626,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T09:26:03.591079+00:00,2026-03-19T09:26:03.643759+00:00


### Windows Manifest (sample)

,source_file,relative_source_path,tensor_npz_path,window_index,frame_start,frame_end_exclusive
0,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,0,0,64
1,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,1,32,96
2,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,2,64,128
3,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,3,96,160
4,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,4,128,192
5,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,5,160,224
6,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,6,192,256
7,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,7,224,288
8,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,8,256,320
9,/home/mitch/development/raccoon-ball/datasets/...,00000000.wav,/home/mitch/development/raccoon-ball/preproces...,9,288,352


## Smoke-Test Tips
- Set `MAX_FILES = 2` to run a quick dry export.
- Or set `SOURCE_FILES = [Path(...), Path(...)]` for targeted normal/abnormal checks.
- Keep `SKIP_EXISTING = True` for restart-safe reruns.
